# Passo a passo do Raio-X: seguir uma empresa e conferir cada etapa

Este notebook segue **uma empresa** pelo caminho inteiro do Raio-X:
tabela de indicadores → grupos (K-means) → empresas parecidas → o que vai para o LLM → texto → validador → raio-x final.

**Como usar:**
1. Rode uma célula por vez (`Shift + Enter`).
2. Antes de cada célula de código, leia o **Preveja** e anote o seu palpite.
3. Depois de rodar, leia o **Confira**. Errou o palpite? Ótimo: é aí que se aprende. Pergunte o porquê.

**Não gasta crédito:** nada aqui chama a API do Gemini.
Funciona com os dados reais (se você tiver a pasta `data/` do Drive, segue a WEG) ou com os dados falsos (segue a ALFA NORTE, código 90001).
Os comandos para preparar o ambiente estão em `aprender/COMANDOS.md`.

In [ ]:
import json
import subprocess
import sys
from pathlib import Path

import pandas as pd

# acha a raiz do repositório (a pasta que tem raiox/), de onde quer que o notebook tenha sido aberto
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "raiox").is_dir())
sys.path.insert(0, str(RAIZ))

from raiox import ia, segmentos

pd.set_option("display.width", 140)
print("raiz do repositório:", RAIZ)

## Passo 1 · A tabela de indicadores (contrato C2)

Uma linha por empresa, com os 6 indicadores calculados pelos scripts do Marcos a partir da DFP da CVM.

**Preveja:** você está usando os dados reais ou os falsos? Quantas empresas tem a tabela?

In [ ]:
df, falsos = segmentos.carregar_indicadores()
print("usando dados falsos?", falsos, "| empresas:", len(df), "| colunas:", len(df.columns))
df.head(3)

**Confira:**
- Dados reais: cerca de 595 empresas, 13 colunas. Dados falsos: 30 empresas, 14 colunas (a tabela falsa tem `perfil_real`).
- Se aparecer **1 coluna só**, o CSV está com `;`. Converta como mostra o `aprender/COMANDOS.md`, seção 3 (até a sessão MA2 entrar).
- Margens e crescimento estão em fração: `0.196` quer dizer 19,6%.

## Passo 2 · Escolher a empresa

**Preveja:** a receita da WEG em 2025 foi maior ou menor que R$ 10 bilhões?

In [ ]:
CD_CVM = 90001 if falsos else 5410      # 5410 = WEG. Troque por outro código para seguir outra empresa
PRODUTO = "Energia Eficiente"
OBJETIVO = "Investigar oportunidades de eficiência energética nas instalações industriais"

empresa = df[df["cd_cvm"] == CD_CVM].iloc[0]
empresa

**Confira:** a receita está em reais. Com os dados reais, a WEG tem cerca de 40,8 bilhões.
Abra o site de relações com investidores da empresa (ri.weg.net) e compare: a diferença deve ser menor que 1%.
Esse é o tipo de conferência que a banca valoriza: **número com fonte, conferido por uma pessoa**.

## Passo 3 · Preparar as variáveis do modelo

Antes do K-means, o código separa as empresas do **treino**: só as com receita a partir de R$ 10 mi (`segmentos.treino()`).
Holdings e empresas sem operação têm margens de −10.000% e distorcem os grupos; elas continuam com raio-x,
mas entram no grupo mais perto depois que o modelo aprendeu.

Depois, `segmentos.matriz()` faz 4 coisas com 5 variáveis:
1. **log da receita**: a receita vai de milhares a centenas de bilhões; sem o log, o tamanho dominaria tudo;
2. **cortar extremos** nos percentis 5 e 95: o que passa disso vira o valor do limite;
3. **nulos viram a mediana**, só para o modelo (a tela continua mostrando "sem dado");
4. **padronizar**: cada coluna passa a ter média 0 e desvio 1, para todas pesarem igual na distância.

**Preveja:** depois de padronizar, qual é a média de cada coluna?

In [ ]:
base = segmentos.treino(df)
print(f"empresas no treino: {len(base)} de {len(df)}")

X = segmentos.variaveis_cruas(base)
print("\nantes de cortar os extremos:")
print(X.describe().loc[["min", "50%", "max"]].round(2))

Xs = segmentos.matriz(base)          # a mesma função que o segmentar() usa
print("\ndepois de cortar e padronizar:")
print(pd.DataFrame(Xs, columns=segmentos.VARIAVEIS).describe().loc[["mean", "std"]].round(2))

**Confira:** média 0 e desvio 1 (ou muito perto disso) em todas as colunas.
Com os dados reais, olhe o `min` e o `max` de `margem_ebit` e `margem_liquida` **antes** do corte: valores de centenas ou
milhares (vezes 100%) vêm de empresas com receita minúscula. Eram a causa do problema da sessão ED4 (próximo passo).

## Passo 4 · K-means e silhueta: quantos grupos?

O K-means testa de 3 a 8 grupos. Para cada número de grupos (k), a **silhueta** dá uma nota de −1 a 1 para a separação.
O código fica com o k de maior silhueta, mas só entre os k cujo menor grupo tem pelo menos 5% das empresas (`segmentos.escolher_k()`).

**Preveja:** os grupos vão ter tamanhos parecidos?

In [ ]:
seg, silhuetas, k = segmentos.segmentar(df)
print("silhueta por k:", {kk: round(v, 2) for kk, v in silhuetas.items()}, "→ k escolhido:", k)
print("empresas por grupo:", seg["grupo"].value_counts().sort_index().to_dict())

**Confira:**
- Com os dados falsos (3 perfis inventados), o k escolhido é 3, com cerca de 10 empresas em cada grupo.
- Com os dados reais de 27/10, um grupo tinha **567 das 595 empresas**. A silhueta alta (0,77) **enganava**: ela só
  separava as poucas empresas extremas.
- Depois do conserto da sessão ED4 (28/10: receita mínima, corte 5/95 e menor grupo com pelo menos 5%), o k é 4, com
  silhueta 0,30 e grupos de 330, 99, 74 e 59 empresas no treino. A silhueta caiu, e os grupos ficaram úteis.

**Lição:** uma métrica boa (silhueta alta) não garante um resultado útil. Sempre olhe o resultado de verdade.

## Passo 5 · O grupo da empresa e as 5 parecidas

As "parecidas" são as 5 empresas mais perto no espaço das variáveis padronizadas (vizinhos mais próximos).

**Preveja:** que tipo de empresa vai aparecer como parecida com a sua?

In [ ]:
linha = seg[seg["cd_cvm"] == CD_CVM].iloc[0]
grupo = seg[seg["grupo"] == linha["grupo"]]
por_cd = seg.set_index("cd_cvm")
print(f"{linha['nome']} está no grupo {linha['grupo']}, que tem {len(grupo)} empresas.")
print("5 parecidas:")
for c in linha["parecidas"]:
    print("  -", por_cd.loc[c, "nome"], "·", por_cd.loc[c, "setor"])

pd.DataFrame({"empresa": linha[segmentos.INDICADORES_NUM].astype(float),
              "mediana do grupo": grupo[segmentos.INDICADORES_NUM].median()})

**Confira:** as parecidas fazem sentido para um vendedor prospectar? Se aparecem empresas de setores sem nenhuma
relação, lembre: as parecidas têm o **mesmo perfil financeiro** (tamanho, crescimento, margens, dívida), não o mesmo setor.
A WEG, por exemplo, fica perto de Cemig e Copel: grandes, lucrativas e pouco endividadas.
O "valor do grupo" é a **mediana** (o valor do meio), que resiste melhor aos extremos que a média.

## Passo 6 · O que vai para o LLM, e o que nunca vai

A função `ia.montar()` separa três coisas:
- `entrada`: **vai para o LLM** (setor, produto, objetivo, indicadores já formatados, posição contra o grupo);
- `valores`: **nunca vai para o LLM**. O código usa para trocar os marcadores (`{margem_ebit}`) pelos números;
- `raio_x`: o que vai para a tela (fontes, números do gráfico, parecidas).

**Preveja:** o nome da empresa vai para o LLM?

In [ ]:
entrada, valores, raio_x = ia.montar(ia._base(), CD_CVM, PRODUTO, OBJETIVO)
print("VAI para o LLM:")
print(json.dumps(entrada, ensure_ascii=False, indent=1)[:1500])
print("\nO nome da empresa aparece no que vai para o LLM?", empresa["nome"] in json.dumps(entrada, ensure_ascii=False))

In [ ]:
print("NUNCA vai para o LLM (o código usa só para trocar os marcadores pelos valores):")
valores

**Confira:** o nome da empresa **não** vai. O texto usa o marcador `{empresa}`, e o código troca no fim.
Isso reduz o risco de o modelo "lembrar" coisas da empresa que não estão nos dados.

## Passo 7 · Acima, abaixo ou em linha: quem decide é o código

Regra: se a diferença entre a empresa e o grupo for de até 3% da mediana, é "em linha"; senão, "acima" ou "abaixo".
O LLM só recebe a posição pronta e não pode mudá-la.

**Preveja:** a margem EBIT da sua empresa está acima ou abaixo do grupo?

In [ ]:
pd.DataFrame(raio_x["indicadores"])[["nome", "valor", "grupo", "posicao", "fonte"]]

**Confira:** cada número tem **fonte** (ano, demonstração e conta da DFP). É isso que a tela mostra em "De onde vem cada número".

## Passo 8 · O texto com marcadores (modo sem IA)

O modo sem IA monta o texto com frases fixas, também com marcadores. É o plano B da demo se a API cair.

**Preveja:** vai aparecer algum número neste texto?

In [ ]:
rx = ia.texto_sem_ia(entrada)
print("resumo:", rx.resumo)
for p in rx.perguntas:
    print("pergunta:", p.texto, f"(base: {p.base})")

**Confira:** nenhum número, só marcadores entre chaves, como `{margem_ebit}`. Com a IA, o Gemini devolve o texto neste mesmo formato.

## Passo 9 · O validador: a regra de ouro na prática

Vamos fingir que o LLM **desobedeceu**: escreveu um número, inventou um marcador e escreveu um ano.

**Preveja:** destas 5 frases, quais sobrevivem ao validador?

In [ ]:
desobediente = ia.RaioX(
    resumo="A {empresa} tem margem EBIT ({margem_ebit}) acima do grupo ({margem_ebit_grupo}).",
    perguntas=[ia.Pergunta(texto="A receita cresceu 26% porque a empresa investiu em fábricas?", base="cresc_2a"),
               ia.Pergunta(texto="Qual é o {lucro_inventado} da área?", base="margem_ebit"),
               ia.Pergunta(texto="Quem decide a compra da solução de {produto}?", base="lacuna")],
    lacunas=["Orçamento para 2027"])

r = ia.validar_e_preencher(desobediente, valores)
print("resumo:", r["resumo"])
print("perguntas que ficaram:", [p["texto"] for p in r["perguntas"]])
print("lacunas que ficaram:", r["lacunas"])
print("removidos:")
for item in r["removidos"]:
    print("  -", item["motivo"], "→", item["texto"])

**Confira:** sobrevivem o resumo e a pergunta sobre quem decide (os marcadores viraram valores).
Caem a pergunta com "26%", a do marcador inventado e a lacuna com "2027".
**É por isso que a IA não consegue mostrar um número inventado na tela:** qualquer dígito fora de marcador derruba a frase.

## Passo 10 · Estragar de propósito: para que servem os testes

E se alguém, sem querer, desligasse a checagem de dígitos? Vamos estragar o validador **só na memória deste notebook**
(nenhum arquivo muda) e ver o que passaria.

**Preveja:** a frase "A margem caiu 3 pontos." passa?

In [ ]:
so_digito = ia.RaioX(resumo="A margem caiu 3 pontos.", perguntas=[], lacunas=[])
checar_original = ia.checar
try:
    ia.checar = lambda texto, permitidos: None          # estraga: aceita qualquer coisa
    print("com o validador estragado:", ia.validar_e_preencher(so_digito, valores)["resumo"])
finally:
    ia.checar = checar_original                          # conserta de volta
print("com o validador certo:    ", repr(ia.validar_e_preencher(so_digito, valores)["resumo"]))

**Confira:** estragado, o número passa; consertado, a frase cai (texto vazio).
Se alguém estragasse o arquivo de verdade, o teste `test_validador_remove_digito_escrito_pela_ia` falharia na hora,
na máquina dele e no CI do PR. Rode os testes do validador:

In [ ]:
saida = subprocess.run([sys.executable, "-m", "pytest", "-q", "--color=no", "-k", "validador"], cwd=RAIZ,
                       capture_output=True, text=True)
print(saida.stdout[-300:])

**Confira:** `5 passed`. Cada erro que o projeto já teve vira um teste que roda para sempre (**teste de regressão**).
A lista dos testes está no `docs/PLANO.md`, seção 4.

## Passo 11 · Tudo junto: o raio-x do contrato C3

A tela chama uma função só: `ia.preparar_raio_x()`. Ela faz os passos 6 a 9 e devolve o raio-x pronto.

**Preveja:** quais chaves o resultado tem?

In [ ]:
final = ia.preparar_raio_x(CD_CVM, PRODUTO, OBJETIVO, usar_ia=False)
print("chaves:", list(final))
print("\nresumo:", final["resumo"])
print("perguntas:")
for p in final["perguntas"]:
    print("  -", p["texto"])
print("lacunas:", final["lacunas"])

**Confira:** as chaves são as do contrato C3 (`contratos.md`), e o resumo já traz os números trocados no lugar dos marcadores.

**Com IA (gasta cerca de US$ 0,0009):** troque `usar_ia=False` por `usar_ia=True`. Precisa do `.env` com a chave.
Rode poucas vezes: mais de 10 chamadas, avise o grupo.

---

## Para fechar: consegue explicar sem olhar?

1. Por que a receita entra em log no modelo?
2. Por que uma silhueta alta pode enganar? (Passo 4)
3. O que vai para o LLM e o que nunca vai? Por quê?
4. Quem decide se um indicador está acima do grupo: o LLM ou o código?
5. Como o validador impede que um número inventado chegue à tela?
6. Para que serve um teste que roda em todo PR?

Se travou em alguma, volte ao passo e peça "explique para iniciante, sem alterar nada".